# 75-10m WSPR Dataset Analysis

This notebook implements the analysis plan from `readme.md` against `7510m_wspr_spots.tsv`.
It covers band openings, distance profiling, geographic spread, SNR vs distance regression, antenna diagnostics, and takeoff angle inference.

Each section includes the intent of the analysis, guidance for interpreting the plots and statistics, and the kinds of conclusions a user can draw from the output.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(context='talk', style='whitegrid', palette='tab10', rc={'figure.dpi': 120})

df = pd.read_csv('7510m_wspr_spots.tsv', sep='	', parse_dates=['Time'])

def get_band(mhz):
    if 3.5 <= mhz <= 4.0:
        return '80m'
    elif 7.0 <= mhz <= 7.3:
        return '40m'
    elif 10.1 <= mhz <= 10.15:
        return '30m'
    elif 14.0 <= mhz <= 14.35:
        return '20m'
    elif 18.068 <= mhz <= 18.168:
        return '17m'
    elif 21.0 <= mhz <= 21.45:
        return '15m'
    elif 24.89 <= mhz <= 24.99:
        return '12m'
    elif 28.0 <= mhz <= 29.7:
        return '10m'
    return 'Other'

df['Band'] = df['MHz'].apply(get_band)
df['rxPrefix'] = df['rxGrid'].fillna('').str[:4]
df['txPrefix'] = df['txGrid'].fillna('').str[:4]
df['TimeBin'] = df['Time'].dt.floor('15min')
df['az_rad'] = np.radians(df['az'].astype(float))

print('Dataset rows:', len(df))
print('Bands present:', df['Band'].value_counts().to_dict())
print('KD3CCO as TX:', (df['TX'] == 'KD3CCO').sum())
print('KD3CCO as RX:', (df['RX'] == 'KD3CCO').sum())


## Analysis 1: Band Openings and Closures

This analysis tracks spot counts and mean SNR across time bins for each band.

Purpose:
- Show when individual amateur bands are most active, and whether propagation is strengthening or weakening over time.
- Highlight changes in the Maximum Usable Frequency (MUF) by comparing multiple bands in the same time window.

How to interpret:
- A rising spot count means the band is opening and more stations are being heard.
- An increasing mean SNR indicates better signal strength and propagation quality.
- Bands that drop sharply suggest closures or fading conditions.

Possible conclusions:
- Identify the best operating times for each band.
- Determine whether the dataset captures a transition from lower-frequency to higher-frequency propagation.
- Spot any bands that remain weak despite others opening, which may suggest local antenna tuning or band-specific absorption.


In [ ]:
rate = df.groupby(['TimeBin', 'Band']).agg(spot_count=('Time', 'count'), mean_snr=('SNR', 'mean')).reset_index()
rate = rate[rate['Band'] != 'Other']

fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
band_order = ['80m','40m','30m','20m','17m','15m','12m','10m']
for band in [b for b in band_order if b in rate['Band'].unique()]:
    band_data = rate[rate['Band'] == band]
    axes[0].plot(band_data['TimeBin'], band_data['spot_count'], label=band, marker='o')
    axes[1].plot(band_data['TimeBin'], band_data['mean_snr'], label=band, marker='o')

axes[0].set_ylabel('Spot Count')
axes[0].set_title('Spot Count by Band and Time Bin')
axes[0].legend(loc='upper left', ncol=4)
axes[1].set_ylabel('Mean SNR (dB)')
axes[1].set_title('Mean SNR by Band and Time Bin')
axes[1].set_xlabel('Time Bin')
fig.autofmt_xdate(rotation=25)
fig.tight_layout()
plt.show()


## Analysis 2: Distance Profiling

This analysis computes mean, maximum, and standard deviation of path distance `k` for every band.

Purpose:
- Quantify how far your station is reaching on each band in the dataset.
- Use the band-specific statistics to compare the effective propagation range across frequencies.

How to interpret:
- Mean distance is the typical path length heard on each band.
- Maximum distance shows the furthest recorded reach and can indicate DX potential.
- Standard deviation reveals how variable the propagation is during the session.

Possible conclusions:
- Bands with higher mean distance are favoring longer skip paths.
- A low standard deviation on a band suggests stable propagation, while a high value indicates mixed local and DX contacts.
- If a high-frequency band has a very small mean distance, the band may be only marginally open.


In [ ]:
distance_stats = df[df['Band'] != 'Other'].groupby('Band')['k'].agg(['mean', 'max', 'std', 'count']).reset_index()
distance_stats['Band'] = pd.Categorical(distance_stats['Band'], categories=['80m','40m','30m','20m','17m','15m','12m','10m'], ordered=True)
distance_stats = distance_stats.sort_values('Band')
distance_stats


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(distance_stats['Band'], distance_stats['mean'], yerr=distance_stats['std'], capsize=6, color=sns.color_palette('tab10', len(distance_stats)))
ax.set_title('Mean Skip Distance by Band with Standard Deviation')
ax.set_xlabel('Band')
ax.set_ylabel('Distance k (km)')
plt.show()


## Analysis 3: Geographical Spread

This analysis identifies the strongest footprint by the top receive grid prefixes.

Purpose:
- Visualize the geographic distribution of contacts by memory of the most active Maidenhead grid areas.
- Understand whether your station is mainly heard domestically, regionally, or in long-distance DX regions.

How to interpret:
- The top grid prefixes show the regions that contribute the largest number of spots.
- A dense domestic footprint suggests strong local and regional propagation.
- Presence of distant grid squares indicates longer skip or transoceanic paths.

Possible conclusions:
- Assess whether the dataset captures mostly local propagation or meaningful DX reach.
- Identify key target regions that the antenna and current conditions favor.
- Use this information to compare with azimuthal coverage and band-specific reach.


In [ ]:
top_grids = df['rxPrefix'].value_counts().head(20).reset_index()
top_grids.columns = ['Grid', 'Count']
top_grids


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_grids, x='Count', y='Grid', palette='viridis', ax=ax)
ax.set_title('Top 20 Receive Grid Prefixes in the Footprint')
ax.set_xlabel('Number of Spots')
ax.set_ylabel('Maidenhead Grid Prefix')
plt.show()


## Analysis 4: SNR vs Distance Regression

This analysis examines path loss trends by plotting SNR against distance for each band.

Purpose:
- Measure how signal strength declines with distance on each band.
- Compare the relative attenuation characteristics across bands.

How to interpret:
- A downward trend is expected: longer distances usually produce lower SNR.
- A tight regression line indicates consistent path loss behavior.
- Scatter far above the trend suggests strong openings or unusually favorable propagation.

Possible conclusions:
- Determine whether some bands are behaving more predictably than others.
- Detect bands where antenna performance or local noise may be affecting SNR independently of distance.
- Spot deviations that could indicate special propagation modes or anomalous paths.


In [ ]:
bands_in_plot = ['80m','40m','30m','20m','17m','15m','12m','10m']
plot_data = df[df['Band'].isin(bands_in_plot)]
fig, axes = plt.subplots(4, 2, figsize=(16, 20), sharex=False, sharey=False)
axes = axes.flatten()
for i, band in enumerate(bands_in_plot):
    band_data = plot_data[plot_data['Band'] == band]
    if len(band_data) > 0:
        sample = band_data.sample(n=min(len(band_data), 800), random_state=1)
        sns.regplot(data=sample, x='k', y='SNR', scatter_kws={'s': 20, 'alpha': 0.6}, lowess=True, ax=axes[i])
    axes[i].set_title(f'SNR vs Distance for {band}')
    axes[i].set_xlabel('Distance k (km)')
    axes[i].set_ylabel('SNR (dB)')
fig.tight_layout()
plt.show()


## Analysis 5: TX vs RX Asymmetry (Local Noise Floor Test)

This comparison uses reciprocal paths involving KD3CCO as both transmitter and receiver.

Purpose:
- Compare how your transmit and receive paths perform for the same remote station and band.
- Reveal whether one direction is consistently stronger, which can indicate local noise, feedline loss, or antenna imbalance.

How to interpret:
- The histogram of `SNR_delta` shows whether TX or RX is generally stronger.
- Values above zero mean TX reports stronger signals than RX.
- Values below zero mean RX reports stronger signals than TX.

Possible conclusions:
- A positive skew suggests the receive path may be suffering from higher local noise or lower sensitivity.
- A negative skew suggests the transmit path may have more loss or less effective radiation.
- A distribution centered near zero indicates roughly symmetric link performance in both directions.


In [ ]:
if 'Band' not in df.columns:
    def get_band(mhz):
        if 3.5 <= mhz <= 4.0:
            return '80m'
        elif 7.0 <= mhz <= 7.3:
            return '40m'
        elif 10.1 <= mhz <= 10.15:
            return '30m'
        elif 14.0 <= mhz <= 14.35:
            return '20m'
        elif 18.068 <= mhz <= 18.168:
            return '17m'
        elif 21.0 <= mhz <= 21.45:
            return '15m'
        elif 24.89 <= mhz <= 24.99:
            return '12m'
        elif 28.0 <= mhz <= 29.7:
            return '10m'
        return 'Other'
    df['Band'] = df['MHz'].apply(get_band)

if 'az_rad' not in df.columns:
    df['az_rad'] = np.radians(df['az'].astype(float))

tx = df[df['TX'] == 'KD3CCO'][['Time', 'Band', 'RX', 'SNR', 'k', 'az']].rename(columns={'RX': 'Remote', 'SNR': 'SNR_tx', 'Time': 'Time_tx', 'k': 'k_tx', 'az': 'az_tx'})
rx = df[df['RX'] == 'KD3CCO'][['Time', 'Band', 'TX', 'SNR', 'k', 'az']].rename(columns={'TX': 'Remote', 'SNR': 'SNR_rx', 'Time': 'Time_rx', 'k': 'k_rx', 'az': 'az_rx'})

paired = pd.merge(rx, tx, on=['Remote', 'Band'], suffixes=('_rx', '_tx'))
paired['time_delta'] = (paired['Time_tx'] - paired['Time_rx']).abs()
paired = paired[paired['time_delta'] <= pd.Timedelta('20min')].copy()
paired = paired.sort_values(['Remote', 'Band', 'Time_rx', 'time_delta']).drop_duplicates(['Remote', 'Band', 'Time_rx'])
paired['SNR_delta'] = paired['SNR_tx'] - paired['SNR_rx']
paired['abs_time_delta_min'] = paired['time_delta'] / pd.Timedelta('1min')
paired[['Remote', 'Band', 'SNR_tx', 'SNR_rx', 'SNR_delta', 'abs_time_delta_min']].head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(paired['SNR_delta'], bins=25, kde=True, color='tab:purple', ax=ax)
ax.axvline(0, color='black', linestyle='--', label='Zero asymmetry')
ax.set_title('TX vs RX SNR Asymmetry for KD3CCO')
ax.set_xlabel('SNR_tx - SNR_rx (dB)')
ax.set_ylabel('Spot count')
ax.legend()
plt.show()


## Analysis 6: Azimuthal Pattern Mapping

This polar map shows spot direction and distance for KD3CCO transmissions.

Purpose:
- Map how signal strength and path length vary with bearing from the station.
- Identify favored antenna lobes and weak nulls in the horizontal plane.

How to interpret:
- Angle corresponds to compass bearing.
- Radius corresponds to path distance.
- Color corresponds to received SNR, so brighter points show stronger paths.

Possible conclusions:
- Strong clusters in certain directions may reveal directional gain or propagation favoring those headings.
- Low-density sectors may indicate nulls or blocked bearings.
- Comparing distance and color helps separate directional propagation from antenna pattern effects.


In [ ]:
tx_local = df[df['TX'] == 'KD3CCO']
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='polar')
sc = ax.scatter(tx_local['az_rad'], tx_local['k'], c=tx_local['SNR'], cmap='viridis', s=35, alpha=0.75)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_title('Azimuthal Radiation Pattern from KD3CCO Transmissions')
cbar = fig.colorbar(sc, ax=ax, pad=0.12)
cbar.set_label('SNR (dB)')
ax.set_rlabel_position(135)
plt.show()


## Analysis 7: Band-by-Band Efficiency Normalization

This analysis compares `k/W` across bands for stations that heard KD3CCO on 3 or more bands.

Purpose:
- Normalize path reach by transmitted power to compare relative efficiency across frequencies.
- Focus on multi-band reference stations to reduce bias from one-off contacts.

How to interpret:
- Higher `k/W` means the station received farther distance for the same power.
- Lower values on a specific band can indicate matching loss, antenna inefficiency, or poor propagation.

Possible conclusions:
- If higher bands show significantly lower `k/W`, the antenna system may be losing efficiency on harmonics.
- Consistent values across bands suggest the matching network and antenna are performing evenly.
- Outliers may point to particular remote stations or directional effects rather than general antenna behavior.


In [ ]:
multi_band = df[df['TX'] == 'KD3CCO'].groupby('RX')['Band'].nunique()
multi_band = multi_band[multi_band >= 3].index
efficiency = df[(df['TX'] == 'KD3CCO') & (df['RX'].isin(multi_band))].copy()
efficiency = efficiency[efficiency['Band'] != 'Other']
efficiency['k_per_watt'] = efficiency['k'] / efficiency['Watts']
fig, ax = plt.subplots(figsize=(14, 8))
sns.boxplot(data=efficiency, x='Band', y='k_per_watt', order=['80m','40m','30m','20m','17m','15m','12m','10m'], ax=ax)
ax.set_title('Normalized Path Efficiency (k/W) by Band for Multi-band Remote Stations')
ax.set_ylabel('Distance per Watt (k/W)')
ax.set_xlabel('Band')
plt.show()


## Analysis 8: Take-Off Angle Inference via Minimum Skip Boundaries

This analysis examines the shortest paths on the higher bands, which informs the likely takeoff angle and near-skip zone.

Purpose:
- Use the lower end of the distance distribution to infer whether the antenna favors low-angle, DX-style radiation or higher-angle local propagation.

How to interpret:
- Shorter 10th and 25th percentile distances imply that the band includes nearer, low-angle paths.
- Larger values suggest the first usable skip is farther away, which may correspond to a higher takeoff angle.

Possible conclusions:
- A small minimum skip boundary is consistent with a low takeoff angle and good near-field performance.
- A large boundary can indicate a high takeoff angle or that the station is primarily hearing longer-range paths.
- Comparing these percentiles across bands helps reveal whether the antenna pattern changes with frequency.


In [ ]:
high_bands = ['20m', '17m', '15m', '12m', '10m']
takeoff = df[df['Band'].isin(high_bands)].groupby('Band')['k'].quantile([0.1, 0.25, 0.5]).unstack()
takeoff = takeoff.reindex(high_bands)
takeoff


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
takeoff.plot(kind='bar', ax=ax)
ax.set_title('Lower Skip Boundary by Higher Band (10th, 25th, and 50th Percentiles)')
ax.set_ylabel('Distance k (km)')
ax.set_xlabel('Band')
plt.xticks(rotation=0)
plt.legend(title='Quantile')
plt.show()
